In [1]:
# Install required packages
!pip install nltk python-Levenshtein matplotlib torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.9/159.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import json
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk
import Levenshtein
from collections import Counter
import math
import random

# Download required NLTK data

nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)


True

In [3]:
def encode_sentence(sentence, token2id, is_urdu=True):
    """Encode a sentence using greedy longest-match subword tokenization."""
    tokens = []

    if is_urdu:
        words = ["_" + w for w in sentence.split()]
    else:
        words = [w + "_" for w in sentence.split()]

    for w in words:
        i = 0
        while i < len(w):
            subword = None
            for j in range(len(w), i, -1):
                piece = w[i:j]
                if piece in token2id:
                    subword = piece
                    break
            if subword is None:
                tokens.append(token2id["<unk>"])
                i += 1
            else:
                tokens.append(token2id[subword])
                i += len(subword)

    return [token2id["<sos>"]] + tokens + [token2id["<eos>"]]



class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab):
        self.src_data = []
        self.tgt_data = []

        for src, tgt in zip(src_sentences, tgt_sentences):
            src_tokens = encode_sentence(src, src_vocab, is_urdu=True)
            tgt_tokens = encode_sentence(tgt, tgt_vocab, is_urdu=False)

            self.src_data.append(torch.tensor(src_tokens))
            self.tgt_data.append(torch.tensor(tgt_tokens))

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return self.src_data[idx], self.tgt_data[idx]


def collate_fn(batch):
    src_batch, tgt_batch = zip(*batch)

    # lengths
    max_src_len = max(len(src) for src in src_batch)
    max_tgt_len = max(len(tgt) for tgt in tgt_batch)

    # we force both to same max length
    max_len = max(max_src_len, max_tgt_len)

    # pad each sequence with 0 up to max_len
    src_padded = [F.pad(src, (0, max_len - len(src)), value=0) for src in src_batch]
    tgt_padded = [F.pad(tgt, (0, max_len - len(tgt)), value=0) for tgt in tgt_batch]

    # stack into batch tensors
    return torch.stack(src_padded), torch.stack(tgt_padded)



In [4]:

# ====================================================================
# Encoder
# ====================================================================
class Encoder(nn.Module):
    def __init__(self, vocab_size=512, embed_dim=128, hidden_dim=128, num_layers=2, dropout=0.1, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)
        self.bilstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True, batch_first=True
        )
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, src):
        """
        Args:
            src: (B, T) padded source sequences

        Returns:
            encoder_outputs: (B, T, 2*H)
            (h, c): (num_layers, B, 2*H)
        """
        embedded = self.dropout(self.embedding(src))
        output, (h, c) = self.bilstm(embedded)
        batch_size = src.size(0)

        # Reshape hidden/cell: (num_layers*2, B, H) → (num_layers, 2, B, H)
        h = h.view(self.num_layers, 2, batch_size, self.hidden_dim)
        c = c.view(self.num_layers, 2, batch_size, self.hidden_dim)

        # Concatenate forward & backward: (num_layers, B, 2*H)
        h = torch.cat((h[:, 0], h[:, 1]), dim=2)
        c = torch.cat((c[:, 0], c[:, 1]), dim=2)

        return output, (h, c)


# ====================================================================
# Decoder
# ====================================================================
class Decoder(nn.Module):
    def __init__(self, vocab_size=512, embed_dim=256, hidden_dim=128, num_layers=4,
                 output_vocab_size=512, dropout=0.3, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.dropout = nn.Dropout(dropout)

        # Decoder expects 2*hidden_dim (from BiLSTM encoder)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim * 2, num_layers,
            dropout=dropout if num_layers > 1 else 0,
            batch_first=True
        )
        self.linear = nn.Linear(hidden_dim * 2, output_vocab_size)
        self.num_layers = num_layers
        self.hidden_dim = hidden_dim

    def forward(self, src_padded, encoder_states):
        """
        Args:
            src_padded: (B, T_dec) - padded src sequence (used as input instead of target)
            encoder_states: (h, c) from encoder (already concatenated)

        Returns:
            output: (B, T_dec, vocab_size)
        """
        embedded = self.dropout(self.embedding(src_padded))
        output, _ = self.lstm(embedded, encoder_states)
        output = self.linear(output)
        return output


# ====================================================================
# Seq2Seq Model
# ====================================================================
class Seq2SeqModel(nn.Module):
    def __init__(self, src_vocab_size=512, tgt_vocab_size=512, embed_dim=128,
                 hidden_dim=128, enc_layers=2, dec_layers=4,
                 enc_dropout=0.1, dec_dropout=0.3, pad_idx=0):
        super().__init__()
        self.encoder = Encoder(src_vocab_size, embed_dim, hidden_dim, enc_layers, enc_dropout, pad_idx)
        self.decoder = Decoder(tgt_vocab_size, embed_dim*2, hidden_dim, dec_layers,
                               tgt_vocab_size, dec_dropout, pad_idx)
        self.pad_idx = pad_idx
        self.enc_layers = enc_layers
        self.dec_layers = dec_layers
        self.hidden_dim = hidden_dim

    def forward(self, src, tgt_len=None):
        batch_size, src_len = src.size()

        # Encode
        enc_output, (h, c) = self.encoder(src)

        # 🔹 Initialize decoder hidden/cell
        # Shape: (dec_layers, batch, hidden_dim*2)
        device = src.device
        dec_hidden = torch.zeros(self.dec_layers, batch_size, self.hidden_dim * 2, device=device)
        dec_cell   = torch.zeros(self.dec_layers, batch_size, self.hidden_dim * 2, device=device)

        # Put encoder’s last layer into decoder’s first layer
        dec_hidden[0] = h[-1]
        dec_cell[0]   = c[-1]

        # Pad src to match decoder length
        if tgt_len is None:
            tgt_len = src_len
        src_padded = torch.full((batch_size, tgt_len), self.pad_idx, device=device)
        src_padded[:, :src_len] = src

        # Decode
        dec_output = self.decoder(src_padded, (dec_hidden, dec_cell))

        return dec_output


In [5]:
def load_data_and_vocab():
    """Load data and vocabularies from Google Drive"""
    base_path = "/content/drive/MyDrive/Model3/"

    # Load source and target sentences
    with open(base_path + "src_normalized.txt", 'r', encoding='utf-8') as f:
        src_sentences = [line.strip() for line in f]

    with open(base_path + "tgt_normalized.txt", 'r', encoding='utf-8') as f:
        tgt_sentences = [line.strip() for line in f]

    # Load vocabularies
    with open(base_path + "vocab_Urdu.json", 'r', encoding='utf-8') as f:
        urdu_vocab = json.load(f)

    with open(base_path + "vocab_Roman.json", 'r', encoding='utf-8') as f:
        roman_vocab = json.load(f)

    return src_sentences, tgt_sentences, urdu_vocab, roman_vocab

def create_datasets(src_sentences, tgt_sentences, urdu_vocab, roman_vocab):
    """Create train/val/test splits"""
    total_size = len(src_sentences)
    train_size = int(0.7 * total_size)
    val_size = int(0.15 * total_size)

    # Shuffle data
    indices = list(range(total_size))
    random.shuffle(indices)

    train_indices = indices[:train_size]
    val_indices = indices[train_size:train_size + val_size]
    test_indices = indices[train_size + val_size:]

    # Create datasets
    train_src = [src_sentences[i] for i in train_indices]
    train_tgt = [tgt_sentences[i] for i in train_indices]

    val_src = [src_sentences[i] for i in val_indices]
    val_tgt = [tgt_sentences[i] for i in val_indices]

    test_src = [src_sentences[i] for i in test_indices]
    test_tgt = [tgt_sentences[i] for i in test_indices]

    train_dataset = TranslationDataset(train_src, train_tgt, urdu_vocab, roman_vocab)
    val_dataset = TranslationDataset(val_src, val_tgt, urdu_vocab, roman_vocab)
    test_dataset = TranslationDataset(test_src, test_tgt, urdu_vocab, roman_vocab)

    return train_dataset, val_dataset, test_dataset

def calculate_perplexity(loss):
    """Calculate perplexity from loss"""
    return math.exp(loss)

def decode_tokens(tokens, id2token):
    """Convert token IDs back to text"""
    words = []
    for token_id in tokens:
        if token_id in [0, 1, 2]:  # pad, sos, eos
            continue
        words.append(id2token.get(token_id, '<unk>'))
    return ' '.join(words)

def calculate_bleu(reference, hypothesis):
    """Calculate BLEU score"""
    reference_tokens = reference.split()
    hypothesis_tokens = hypothesis.split()

    if len(hypothesis_tokens) == 0:
        return 0.0

    smoothie = SmoothingFunction().method4
    return sentence_bleu([reference_tokens], hypothesis_tokens, smoothing_function=smoothie)

def calculate_cer(reference, hypothesis):
    """Calculate Character Error Rate"""
    if len(reference) == 0:
        return 1.0 if len(hypothesis) > 0 else 0.0
    return Levenshtein.distance(reference, hypothesis) / len(reference)

def calculate_edit_distance(reference, hypothesis):
    """Calculate Levenshtein distance"""
    return Levenshtein.distance(reference, hypothesis)


In [6]:
def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device, roman_vocab):
    id2roman = {v: k for k, v in roman_vocab.items()}

    for epoch in range(epochs):
        # ---- TRAINING ----
        model.train()
        total_loss = 0
        correct, total = 0, 0

        for batch_idx, (src, tgt) in enumerate(train_loader):
            src, tgt = src.to(device), tgt.to(device)

            optimizer.zero_grad()
            output = model(src)  # (batch, seq_len, vocab_size)

            # Flatten for CE Loss
            output_flat = output.reshape(-1, output.size(-1))
            tgt_flat = tgt.reshape(-1)

            loss = criterion(output_flat, tgt_flat)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += loss.item()

            # Accuracy (token-level)
            predictions = torch.argmax(output, dim=-1)
            correct += (predictions == tgt).sum().item()
            total += tgt.numel()

        avg_train_loss = total_loss / len(train_loader)
        train_perplexity = calculate_perplexity(avg_train_loss)
        train_accuracy = correct / total

        # ---- VALIDATION ----
        model.eval()
        val_loss, val_bleu, val_cer, val_edit, val_correct, val_total = 0, 0, 0, 0, 0, 0

        with torch.no_grad():
            for src, tgt in val_loader:
                src, tgt = src.to(device), tgt.to(device)
                output = model(src)

                # Loss
                output_flat = output.reshape(-1, output.size(-1))
                tgt_flat = tgt.reshape(-1)
                loss = criterion(output_flat, tgt_flat)
                val_loss += loss.item()

                # Predictions
                predictions = torch.argmax(output, dim=-1)
                val_correct += (predictions == tgt).sum().item()
                val_total += tgt.numel()

                for i in range(src.size(0)):
                    pred_tokens = predictions[i].cpu().numpy()
                    tgt_tokens = tgt[i].cpu().numpy()

                    pred_text = decode_tokens(pred_tokens, id2roman)
                    ref_text = decode_tokens(tgt_tokens, id2roman)

                    val_bleu += calculate_bleu(ref_text, pred_text)
                    val_cer += calculate_cer(ref_text, pred_text)
                    val_edit += calculate_edit_distance(ref_text, pred_text)

        # Averages
        avg_val_loss = val_loss / len(val_loader)
        val_perplexity = calculate_perplexity(avg_val_loss)
        avg_val_bleu = val_bleu / len(val_loader.dataset)
        avg_val_cer = val_cer / len(val_loader.dataset)
        avg_val_edit = val_edit / len(val_loader.dataset)
        val_accuracy = val_correct / val_total

        # ---- RESULTS ----
        print(f"\nEpoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f}, Perplexity: {train_perplexity:.4f}, Accuracy: {train_accuracy:.4f}")
        print(f"Val Loss:   {avg_val_loss:.4f}, Perplexity: {val_perplexity:.4f}, "
              f"Accuracy: {val_accuracy:.4f}, BLEU: {avg_val_bleu:.4f}, CER: {avg_val_cer:.4f}, "
              f"Edit Dist: {avg_val_edit:.2f}")




def evaluate_model(model, test_loader, roman_vocab, device):
    """Evaluate model with metrics"""
    model.eval()
    id2roman = {v: k for k, v in roman_vocab.items()}

    total_bleu = 0
    total_cer = 0
    total_edit_dist = 0
    total_loss = 0
    count = 0

    criterion = nn.CrossEntropyLoss(ignore_index=0)

    with torch.no_grad():
        for src, tgt in test_loader:
            src, tgt = src.to(device), tgt.to(device)

            output = model(src)

            # Calculate loss
            loss_output_flat = output.reshape(-1, output.size(-1))
            target_flat = tgt.reshape(-1)
            loss = criterion(loss_output_flat, target_flat)
            total_loss += loss.item()

            predictions = torch.argmax(output, dim=-1)

            for i in range(src.size(0)):
                pred_tokens = predictions[i].cpu().numpy()
                tgt_tokens = tgt[i].cpu().numpy()

                pred_text = decode_tokens(pred_tokens, id2roman)
                ref_text = decode_tokens(tgt_tokens, id2roman)

                total_bleu += calculate_bleu(ref_text, pred_text)
                total_cer += calculate_cer(ref_text, pred_text)
                total_edit_dist += calculate_edit_distance(ref_text, pred_text)
                count += 1

    avg_loss = total_loss / len(test_loader)
    avg_bleu = total_bleu / count
    avg_cer = total_cer / count
    avg_edit_dist = total_edit_dist / count

    print(f"Test Loss: {avg_loss:.4f}, Perplexity: {calculate_perplexity(avg_loss):.4f}")
    print(f"BLEU Score: {avg_bleu:.4f}")
    print(f"Character Error Rate: {avg_cer:.4f}")
    print(f"Average Edit Distance: {avg_edit_dist:.2f}")

    return avg_bleu, avg_cer, avg_edit_dist

def show_examples(model, test_dataset, urdu_vocab, roman_vocab, device, num_examples=5):
    """Show translation examples"""
    model.eval()
    id2roman = {v: k for k, v in roman_vocab.items()}
    id2urdu = {v: k for k, v in urdu_vocab.items()}

    indices = random.sample(range(len(test_dataset)), num_examples)

    with torch.no_grad():
        for idx in indices:
            src, tgt = test_dataset[idx]
            src_batch = src.unsqueeze(0).to(device)

            output = model(src_batch)
            prediction = torch.argmax(output, dim=-1).squeeze(0)

            src_text = decode_tokens(src.numpy(), id2urdu)
            tgt_text = decode_tokens(tgt.numpy(), id2roman)
            pred_text = decode_tokens(prediction.cpu().numpy(), id2roman)

            print(f"Source (Urdu): {src_text}")
            print(f"Target (Roman): {tgt_text}")
            print(f"Prediction: {pred_text}")

            # Calculate metrics for this example
            bleu = calculate_bleu(tgt_text, pred_text)
            cer = calculate_cer(tgt_text, pred_text)
            edit_dist = calculate_edit_distance(tgt_text, pred_text)

            print(f"BLEU: {bleu:.3f}, CER: {cer:.3f}, Edit Dist: {edit_dist}")
            print("-" * 50)


In [7]:
def test_metrics_sanity_check():
    """Sanity check for evaluation metrics"""
    print("Testing evaluation metrics...")

    # Test BLEU
    ref = "yeh ek test sentence hai"
    hyp = "yeh ek test sentence hai"
    print(f"BLEU (identical): {calculate_bleu(ref, hyp):.4f}")

    hyp = "yeh test sentence hai"
    print(f"BLEU (missing word): {calculate_bleu(ref, hyp):.4f}")

    # Test CER
    ref = "hello world"
    hyp = "hello world"
    print(f"CER (identical): {calculate_cer(ref, hyp):.4f}")

    hyp = "helo wrold"
    print(f"CER (2 errors): {calculate_cer(ref, hyp):.4f}")

    # Test Edit Distance
    print(f"Edit distance (identical): {calculate_edit_distance('hello', 'hello')}")
    print(f"Edit distance (1 substitution): {calculate_edit_distance('hello', 'hallo')}")


In [8]:
test_metrics_sanity_check()

Testing evaluation metrics...
BLEU (identical): 1.0000
BLEU (missing word): 0.3611
CER (identical): 0.0000
CER (2 errors): 0.2727
Edit distance (identical): 0
Edit distance (1 substitution): 1


In [9]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cuda


In [10]:
# Load data
src_sentences, tgt_sentences, urdu_vocab, roman_vocab = load_data_and_vocab()


In [11]:
# Create datasets
train_dataset, val_dataset, test_dataset = create_datasets(
    src_sentences, tgt_sentences, urdu_vocab, roman_vocab)


In [12]:
# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)


In [13]:

# Initialize model
model = Seq2SeqModel().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding


In [14]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _مح فل یں _بر ہ م _کر ے _ہے _گ نج ف ہ _با ز _خیال
Target (Roman): ma h fi le n_ ba r ham_ kare_ hai_ ga n ji fa - baz -e- khayal_
Prediction: ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e-
BLEU: 0.000, CER: 1.524, Edit Dist: 96
--------------------------------------------------
Source (Urdu): _خ نج ر _ن کا ل _دل _میں _اگر _ا م ت ح اں _کی _ہے
Target (Roman): kh an ja r_ nik a l_ dil_ men_ agar_ i m ti ha n_ ki_ hai_
Prediction: ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e-
BLEU: 0.000, CER: 1.724, Edit Dist: 100
--------------------------------------------------
Source (Urdu): _ک ھ نچ _کے _ر گ _ر گ _میں _مر ے _نش ت ر _ف ص اد _آیا
Target (Roman): kh in ch _ ke_ ra g_ ra g_ men_ mire_ na sh ta r-e- fa s sa d_ aaya_
Prediction: ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e-

In [15]:
print("Starting training...")

# Step 1: Train encoder only (freeze decoder)
print("\nStep 1: Training encoder (10 epochs)")
for param in model.decoder.parameters():
    param.requires_grad = False

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)

train_model(model, train_loader, val_loader, criterion, optimizer, 10, device, roman_vocab)


Starting training...

Step 1: Training encoder (5 epochs)

Epoch 1/10
Train Loss: 6.2346, Perplexity: 510.1026, Accuracy: 0.0003
Val Loss:   6.2341, Perplexity: 509.8347, Accuracy: 0.0002, BLEU: 0.0000, CER: 3.0197, Edit Dist: 161.07

Epoch 2/10
Train Loss: 6.2344, Perplexity: 510.0016, Accuracy: 0.0003
Val Loss:   6.2340, Perplexity: 509.7654, Accuracy: 0.0002, BLEU: 0.0000, CER: 3.0197, Edit Dist: 161.07

Epoch 3/10
Train Loss: 6.2342, Perplexity: 509.8981, Accuracy: 0.0003
Val Loss:   6.2337, Perplexity: 509.6459, Accuracy: 0.0002, BLEU: 0.0000, CER: 3.0196, Edit Dist: 161.06

Epoch 4/10
Train Loss: 6.2340, Perplexity: 509.7876, Accuracy: 0.0004
Val Loss:   6.2335, Perplexity: 509.5180, Accuracy: 0.0002, BLEU: 0.0001, CER: 3.0181, Edit Dist: 160.98

Epoch 5/10
Train Loss: 6.2337, Perplexity: 509.6465, Accuracy: 0.0005
Val Loss:   6.2332, Perplexity: 509.3744, Accuracy: 0.0005, BLEU: 0.0002, CER: 3.0057, Edit Dist: 160.30

Epoch 6/10
Train Loss: 6.2335, Perplexity: 509.5201, Accuracy

In [16]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _اٹھا ئے _اٹ ھ _نہیں _سک تا _یہ _درد _سر _پھر _بھی
Target (Roman): ut ha e_ ut h_ nahin_ sak ta _ ye_ da r d-e- sa r_ phir_ bhi_
Prediction: ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e-
BLEU: 0.000, CER: 1.213, Edit Dist: 74
--------------------------------------------------
Source (Urdu): _دل _میں _کت نے _خوش _تھے _اپنی _فر قت _کی _آ را ئ ش _پر
Target (Roman): dil_ men_ ki t ne_ kh u sh _ the_ apni_ fur qat_ ki_ ar a is h_ pa r_
Prediction: ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e-
BLEU: 0.000, CER: 1.333, Edit Dist: 92
--------------------------------------------------
Source (Urdu): _یہ _م یر ؔ _کا _دی وا ن _یہاں _بھی _ہے _وہ اں _بھی
Target (Roman): ye_ ' mi r '_ ka_ di va n_ ya ha n_ bhi_ hai_ va ha n_ bhi_
Prediction: ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- ahl-e- a

In [17]:
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (10 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
train_model(model, train_loader, val_loader, criterion, optimizer, 15, device,roman_vocab)


# Epoch 15/15
# Train Loss: 2.0447, Perplexity: 7.7265, Accuracy: 0.3342
# Val Loss:   2.1611, Perplexity: 8.6804, Accuracy: 0.3366, BLEU: 0.3020, CER: 0.2647, Edit Dist: 15.02



Step 2: Training decoder (10 epochs)

Epoch 1/15
Train Loss: 4.9202, Perplexity: 137.0341, Accuracy: 0.0869
Val Loss:   4.8099, Perplexity: 122.7232, Accuracy: 0.0922, BLEU: 0.0123, CER: 0.6598, Edit Dist: 36.87

Epoch 2/15
Train Loss: 4.6760, Perplexity: 107.3351, Accuracy: 0.0976
Val Loss:   4.5344, Perplexity: 93.1680, Accuracy: 0.1027, BLEU: 0.0181, CER: 0.5877, Edit Dist: 32.88

Epoch 3/15
Train Loss: 4.4175, Perplexity: 82.8885, Accuracy: 0.1108
Val Loss:   4.2328, Perplexity: 68.9133, Accuracy: 0.1234, BLEU: 0.0266, CER: 0.5368, Edit Dist: 29.97

Epoch 4/15
Train Loss: 4.0745, Perplexity: 58.8204, Accuracy: 0.1383
Val Loss:   3.8583, Perplexity: 47.3830, Accuracy: 0.1582, BLEU: 0.0430, CER: 0.4863, Edit Dist: 27.23

Epoch 5/15
Train Loss: 3.7625, Perplexity: 43.0557, Accuracy: 0.1656
Val Loss:   3.5634, Perplexity: 35.2836, Accuracy: 0.1866, BLEU: 0.0631, CER: 0.4456, Edit Dist: 24.98

Epoch 6/15
Train Loss: 3.5023, Perplexity: 33.1930, Accuracy: 0.1880
Val Loss:   3.3295, Perp

In [18]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _اپنی _مح ر وم یا ں _چ ھ پا تے _ہیں
Target (Roman): apni_ ma h ru mi ya n_ chhu pa te_ hain_
Prediction: apni_ ma h r ma ya n_ chhu pa te_ hain_
BLEU: 0.588, CER: 0.050, Edit Dist: 2
--------------------------------------------------
Source (Urdu): _لگ ا _کے _دیکھ _ل ے _جو _بھی _حس اب _آتا _ہو
Target (Roman): lag a_ ke_ dekh_ le_ jo_ bhi_ hi sa b_ aata_ ho_
Prediction: lag a_ ke_ dekh liya_ ke_ ka bhi_ ra b_ hota_ ho_
BLEU: 0.126, CER: 0.312, Edit Dist: 15
--------------------------------------------------
Source (Urdu): _کی ا _کر یں _وہ _سن ان ے _کو _پی ار _کی _بات یں
Target (Roman): kiya_ ka re n_ vo_ su na ne_ ko_ pyaar_ ki_ baten_
Prediction: kya_ hai_ r_ n_ vo_ sa n e_ ko_ pyaar_ pyaar_ ki_
BLEU: 0.079, CER: 0.340, Edit Dist: 17
--------------------------------------------------
Source (Urdu): _کبھی _ہے _دو ری _کبھی _مل ن _ہے _جی تے _جا ؤ _سوچ و _م ت
Target (Roman): ka bhi_ hai_ du u ri_ ka bhi_ mil an _ hai_ ji it e_ ja a o_ so ch o_ mat_
Pre

In [19]:
#more 10 epochs
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (10 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
train_model(model, train_loader, val_loader, criterion, optimizer, 10, device,roman_vocab)
# Epoch 10/10
# Train Loss: 1.5777, Perplexity: 4.8438, Accuracy: 0.3983
# Val Loss:   1.9996, Perplexity: 7.3857, Accuracy: 0.3726, BLEU: 0.3823, CER: 0.2259, Edit Dist: 12.81


Step 2: Training decoder (10 epochs)

Epoch 1/10
Train Loss: 2.2632, Perplexity: 9.6140, Accuracy: 0.3113
Val Loss:   2.3436, Perplexity: 10.4189, Accuracy: 0.3101, BLEU: 0.2592, CER: 0.2961, Edit Dist: 16.54

Epoch 2/10
Train Loss: 2.2020, Perplexity: 9.0429, Accuracy: 0.3178
Val Loss:   2.2998, Perplexity: 9.9726, Accuracy: 0.3173, BLEU: 0.2750, CER: 0.2896, Edit Dist: 16.13

Epoch 3/10
Train Loss: 2.1477, Perplexity: 8.5653, Accuracy: 0.3263
Val Loss:   2.2848, Perplexity: 9.8240, Accuracy: 0.3200, BLEU: 0.2850, CER: 0.2839, Edit Dist: 15.87

Epoch 4/10
Train Loss: 2.0951, Perplexity: 8.1266, Accuracy: 0.3317
Val Loss:   2.2967, Perplexity: 9.9411, Accuracy: 0.3190, BLEU: 0.2870, CER: 0.2846, Edit Dist: 15.88

Epoch 5/10
Train Loss: 2.0522, Perplexity: 7.7849, Accuracy: 0.3369
Val Loss:   2.2236, Perplexity: 9.2408, Accuracy: 0.3273, BLEU: 0.2987, CER: 0.2792, Edit Dist: 15.63

Epoch 6/10
Train Loss: 2.0074, Perplexity: 7.4443, Accuracy: 0.3443
Val Loss:   2.2395, Perplexity: 9.388

In [20]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _تو _بھی _خوش بو _ہے _مگر _میرا _ت ج س س _بے _کا ر
Target (Roman): tu_ bhi_ kh u sh bu _ hai_ magar_ mer a_ ta ja s su s_ be ka r_
Prediction: tu_ bhi_ kh u sh bu _ hai_ mer mer a_ a_ n s _
BLEU: 0.408, CER: 0.317, Edit Dist: 20
--------------------------------------------------
Source (Urdu): _نہیں _م ط لب _ایک _کو _ایک _سے _یہ _ا د ھر _چل ا _وہ _ا د ھر _گیا
Target (Roman): nahin_ ma t la b_ ek_ ko_ ek_ se_ ye_ i dhar_ ch a la _ vo_ u dhar_ gaya_
Prediction: nahin_ ma t la b_ ek_ ek_ ek_ se_ ye_ a dhar_ ch a la _ vo_ dhar_ dhar_ gaya_
BLEU: 0.566, CER: 0.110, Edit Dist: 8
--------------------------------------------------
Source (Urdu): _صبح _موج _ن سی م _گل شن _کو
Target (Roman): subh_ mau j_ na si m_ gu l sh an _ ko_
Prediction: subh_ mau -e- -e- na na gu gu l
BLEU: 0.068, CER: 0.579, Edit Dist: 22
--------------------------------------------------
Source (Urdu): _اپنے _ح ق _میں _دع ا _کر ے _کوئی
Target (Roman): apne_ ha q_ men_ du a_ kare_ koi_

In [21]:
# Step 3: Train full model
print("\nStep 3: Training full model (10 epochs)")
for param in model.parameters():
    param.requires_grad = True

optimizer = optim.Adam(model.parameters(), lr=0.0005)
train_model(model, train_loader, val_loader, criterion, optimizer, 10, device,roman_vocab)



Step 3: Training full model (10 epochs)

Epoch 1/10
Train Loss: 1.7878, Perplexity: 5.9763, Accuracy: 0.3737
Val Loss:   2.1582, Perplexity: 8.6552, Accuracy: 0.3447, BLEU: 0.3428, CER: 0.2611, Edit Dist: 14.45

Epoch 2/10
Train Loss: 1.7502, Perplexity: 5.7559, Accuracy: 0.3807
Val Loss:   2.1344, Perplexity: 8.4523, Accuracy: 0.3483, BLEU: 0.3495, CER: 0.2584, Edit Dist: 14.27

Epoch 3/10
Train Loss: 1.7290, Perplexity: 5.6351, Accuracy: 0.3824
Val Loss:   2.1294, Perplexity: 8.4101, Accuracy: 0.3498, BLEU: 0.3511, CER: 0.2604, Edit Dist: 14.33

Epoch 4/10
Train Loss: 1.7052, Perplexity: 5.5025, Accuracy: 0.3840
Val Loss:   2.1439, Perplexity: 8.5327, Accuracy: 0.3493, BLEU: 0.3513, CER: 0.2633, Edit Dist: 14.44

Epoch 5/10
Train Loss: 1.6826, Perplexity: 5.3794, Accuracy: 0.3863
Val Loss:   2.1347, Perplexity: 8.4544, Accuracy: 0.3492, BLEU: 0.3482, CER: 0.2713, Edit Dist: 14.86

Epoch 6/10
Train Loss: 1.6624, Perplexity: 5.2717, Accuracy: 0.3904
Val Loss:   2.1107, Perplexity: 8.2

In [22]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _پر د ۂ _عر ض _وفا _میں _بھی _رہا _ہو ں _کر تا
Target (Roman): pa r da-e- ar z-e- vafa_ men_ bhi_ ra ha _ huun_ ka r ta _
Prediction: pa r da-e- ar z-e- vafa_ men_ bhi_ ra ha _ huun_ ka r
BLEU: 0.867, CER: 0.086, Edit Dist: 5
--------------------------------------------------
Source (Urdu): _عشق _تری _انت ہ ا _عشق _مر ی _انت ہ ا
Target (Roman): ishq_ tiri_ inti ha _ ishq_ miri_ inti ha _
Prediction: ishq_ tiri_ inti ha _ ishq_ ma fa r _ _
BLEU: 0.480, CER: 0.233, Edit Dist: 10
--------------------------------------------------
Source (Urdu): _ح ص و ل _پر _مجھ ے _اس _درد _سر _سے _کچھ _نہ _ہوا
Target (Roman): hus ul_ pa r_ mujhe_ us_ da r d-e- sa r_ se_ kuchh_ na_ hua_
Prediction: ha s s l_ pa r_ mujhe_ us_ dard_ sa sa r_ se_ kuchh_ na_
BLEU: 0.412, CER: 0.250, Edit Dist: 15
--------------------------------------------------
Source (Urdu): _جسے _عشق _کا _ت یر _کا ری _لگ ے
Target (Roman): ji se_ ishq_ ka_ ti ir _ ka ar i_ lag e_
Prediction: ji se_ ish

In [23]:
# Step 4: Fine-tune with low learning rate and decay
print("\nStep 4: Fine-tuning with learning rate decay (10 epochs)")
optimizer = optim.Adam(model.parameters(), lr=0.0001)
scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9)

for epoch in range(10):
    train_model(model, train_loader, val_loader, criterion, optimizer, 1, device, roman_vocab)
    scheduler.step()
    print(f"Learning rate: {scheduler.get_last_lr()[0]:.6f}")
# Epoch 1/1
# Train Loss: 0.9506, Perplexity: 2.5873, Accuracy: 0.4920
# Val Loss:   1.9914, Perplexity: 7.3255, Accuracy: 0.4026, BLEU: 0.4508, CER: 0.1951, Edit Dist: 11.02
# Learning rate: 0.000035


Step 4: Fine-tuning with learning rate decay (10 epochs)

Epoch 1/1
Train Loss: 1.5488, Perplexity: 4.7059, Accuracy: 0.4066
Val Loss:   2.0833, Perplexity: 8.0311, Accuracy: 0.3612, BLEU: 0.3689, CER: 0.2704, Edit Dist: 14.67
Learning rate: 0.000090

Epoch 1/1
Train Loss: 1.5317, Perplexity: 4.6260, Accuracy: 0.4100
Val Loss:   2.1017, Perplexity: 8.1803, Accuracy: 0.3618, BLEU: 0.3716, CER: 0.2681, Edit Dist: 14.53
Learning rate: 0.000081

Epoch 1/1
Train Loss: 1.5199, Perplexity: 4.5718, Accuracy: 0.4110
Val Loss:   2.0914, Perplexity: 8.0959, Accuracy: 0.3618, BLEU: 0.3708, CER: 0.2703, Edit Dist: 14.65
Learning rate: 0.000073

Epoch 1/1
Train Loss: 1.5135, Perplexity: 4.5427, Accuracy: 0.4137
Val Loss:   2.0991, Perplexity: 8.1585, Accuracy: 0.3615, BLEU: 0.3715, CER: 0.2696, Edit Dist: 14.61
Learning rate: 0.000066

Epoch 1/1
Train Loss: 1.5050, Perplexity: 4.5041, Accuracy: 0.4144
Val Loss:   2.0979, Perplexity: 8.1493, Accuracy: 0.3625, BLEU: 0.3720, CER: 0.2700, Edit Dist: 14

In [24]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _اب _بھی _گ ر _پڑ _کے _ض ع ف _سے _ن ال ے
Target (Roman): ab_ bhi_ gi r_ pa d_ ke_ z o f_ se_ na a le_
Prediction: ab_ bhi_ gu r_ pa _ ke_ za o af se_ se_ a le_
BLEU: 0.079, CER: 0.182, Edit Dist: 8
--------------------------------------------------
Source (Urdu): _یہی _وہ _تھے _جن ہ یں _ہن س _ہن س _کے _جا ن _دی نا _تھا
Target (Roman): ya hi_ vo_ the_ ji n hen_ ha n s_ ha n s_ ke_ ja an _ de na_ tha_
Prediction: ya hi_ vo_ the_ ji hen_ hen_ ha n s_ ha n s_ ke_ ja an _
BLEU: 0.694, CER: 0.231, Edit Dist: 15
--------------------------------------------------
Source (Urdu): _غ ض ب _کی ا _ترے _و عد ے _پہ _ا عت بار _کی ا
Target (Roman): gha za b_ kiya_ tire_ va a de _ pe_ e ' ti ba r_ kiya_
Prediction: gha za b_ ki_ hai_ ka va d_ _ pe_ un ti ti ba r_ kiya_
BLEU: 0.253, CER: 0.241, Edit Dist: 13
--------------------------------------------------
Source (Urdu): _ہن س _کے _ب لو ا ئی ے _م ٹ _جائے _گا _سب _دل _کا _گل ہ
Target (Roman): ha n s_ ke_ bu l va i ye

In [25]:
# Evaluation
print("\nEvaluation Results:")
evaluate_model(model, test_loader, roman_vocab, device)
# Evaluation Results:
# Test Loss: 2.0089, Perplexity: 7.4549
# BLEU Score: 0.4518
# Character Error Rate: 0.1949
# Average Edit Distance: 10.96


Evaluation Results:
Test Loss: 2.0479, Perplexity: 7.7520
BLEU Score: 0.3851
Character Error Rate: 0.2593
Average Edit Distance: 14.12


(0.3850716844062374, 0.2592933512620931, 14.120805369127517)

In [28]:
#more 10 epochs
# Step 2: Train decoder only (freeze encoder)
print("\nStep 2: Training decoder (10 epochs)")
for param in model.encoder.parameters():
    param.requires_grad = False
for param in model.decoder.parameters():
    param.requires_grad = True

optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)
train_model(model, train_loader, val_loader, criterion, optimizer, 10, device,roman_vocab)


Step 2: Training decoder (10 epochs)

Epoch 1/10
Train Loss: 1.6115, Perplexity: 5.0105, Accuracy: 0.3994
Val Loss:   2.1773, Perplexity: 8.8228, Accuracy: 0.3527, BLEU: 0.3611, CER: 0.2805, Edit Dist: 15.19

Epoch 2/10
Train Loss: 1.6192, Perplexity: 5.0492, Accuracy: 0.3967
Val Loss:   2.1133, Perplexity: 8.2752, Accuracy: 0.3578, BLEU: 0.3640, CER: 0.2892, Edit Dist: 15.54

Epoch 3/10
Train Loss: 1.6138, Perplexity: 5.0217, Accuracy: 0.3972
Val Loss:   2.1127, Perplexity: 8.2702, Accuracy: 0.3563, BLEU: 0.3555, CER: 0.2947, Edit Dist: 15.88

Epoch 4/10
Train Loss: 1.5906, Perplexity: 4.9068, Accuracy: 0.4000
Val Loss:   2.1217, Perplexity: 8.3457, Accuracy: 0.3566, BLEU: 0.3557, CER: 0.2856, Edit Dist: 15.48

Epoch 5/10
Train Loss: 1.5728, Perplexity: 4.8203, Accuracy: 0.4019
Val Loss:   2.0966, Perplexity: 8.1387, Accuracy: 0.3609, BLEU: 0.3611, CER: 0.2949, Edit Dist: 15.87

Epoch 6/10
Train Loss: 1.5645, Perplexity: 4.7801, Accuracy: 0.4032
Val Loss:   2.1260, Perplexity: 8.3810

In [29]:
# Evaluation
print("\nEvaluation Results:")
evaluate_model(model, test_loader, roman_vocab, device)


Evaluation Results:
Test Loss: 2.0439, Perplexity: 7.7209
BLEU Score: 0.3836
Character Error Rate: 0.2742
Average Edit Distance: 14.85


(0.3836170291601699, 0.27418701813135116, 14.847235538510706)

In [30]:
# Show examples
print("\nTranslation Examples:")
show_examples(model, test_dataset, urdu_vocab, roman_vocab, device)



Translation Examples:
Source (Urdu): _چمن _اور _بھی _آش یا ں _اور _بھی _ہیں
Target (Roman): chaman_ aur_ bhi_ ash i ya n_ aur_ bhi_ hain_
Prediction: chaman_ aur_ bhi_ ash ya ya n_ aur_ bhi_ hain_
BLEU: 0.658, CER: 0.044, Edit Dist: 2
--------------------------------------------------
Source (Urdu): _ص نم _ت یر ے _ن ی ن _کی _آ ر زو _میں
Target (Roman): sa nam_ ter e_ na ya n_ ki_ aa r zu_ men_
Prediction: sa nam_ ta e_ ta a _ _ a _
BLEU: 0.052, CER: 0.415, Edit Dist: 17
--------------------------------------------------
Source (Urdu): _ا ن _سے _نظر یں _کی ا _مل یں _رو شن _ف ضا ئی ں _ہو _گئی ں
Target (Roman): un_ se_ naz re n_ kya_ mil in_ rau sh an _ fa za en_ ho_ ga iin_
Prediction: un_ sa bhi_ hai_ n_ kya_ hai_ ti_ ya ra sh _ fa zi en_ ho_ i_ iin_
BLEU: 0.063, CER: 0.375, Edit Dist: 24
--------------------------------------------------
Source (Urdu): _یہ _جو ش _گ ری ہ _تو _دیکھ و _کہ _جب _فر قت _میں _رو یا _ہو ں
Target (Roman): ye_ jo sh-e- gi r ya_ to_ dekh o_ ki_ ja b_ fur qat_ me

In [31]:
save_path = "/content/drive/MyDrive/Model3/urdu_roman_nmt_model.pth"
# Save model
torch.save({
    'model_state_dict': model.state_dict(),
    'urdu_vocab': urdu_vocab,
    'roman_vocab': roman_vocab
}, save_path)
print("Model saved as 'urdu_roman_nmt_model.pth'")



Model saved as 'urdu_roman_nmt_model.pth'
